# Sistema de Recomendación - Tu Futuro según la Data
predice si una persona ganará más o menos de 50,000 USD al año,
basándose en sus datos demográficos y socioeconómicos.


## Paso 1: Importar librerías
Importamos todas las herramientas que vamos a necesitar.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

## Paso 2: Cargar y limpiar los datos
Cargamos el dataset del censo. Limpiamos los valores '?' que significan datos desconocidos,
y creamos una columna nueva: 1 si gana más de 50K, 0 si gana menos.

In [2]:
# Cargar el dataset
df = pd.read_csv("https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv")

# Reemplazar '?' por valores nulos y eliminar esas filas
df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)

# Limpiar espacios en la columna income
df['income'] = df['income'].str.strip()

# Crear columna objetivo: 1 = gana más de 50K, 0 = gana menos
df['high_income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

# Ver las primeras filas
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,high_income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,0
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,0
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,0
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K,0
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K,0


## Paso 3: Seleccionar variables y preprocesar
Elegimos las columnas que usará el modelo.
- Variables numéricas: edad y horas trabajadas por semana
- Variables categóricas: educación, estado civil, ocupación, sexo, país

In [3]:
# Columnas que usaremos como entrada del modelo
features = ['age', 'education', 'marital.status', 'occupation', 'hours.per.week', 'sex', 'native.country']
X = df[features]
y = df['high_income']

# Separar variables numéricas y categóricas
numeric_features = ['age', 'hours.per.week']
categorical_features = ['education', 'marital.status', 'occupation', 'sex', 'native.country']

# Normalizar números y convertir categorías a números
numeric_transformer = StandardScaler()  # Normaliza números (ej: edad)
categorical_transformer = OneHotEncoder(handle_unknown='ignore')  # Convierte texto a números

# Combinar los dos transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Paso 4: Crear y entrenar el modelo
Usamos Regresión Logística dentro de un Pipeline.
Un Pipeline encadena el preprocesamiento y el modelo en un solo paso.

In [4]:
# Crear el pipeline: preprocesamiento + modelo
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Dividir datos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar el modelo
model.fit(X_train, y_train)

print(f"✅ Modelo entrenado con {len(X_train)} personas")
print(f"📊 Precisión en datos de prueba: {model.score(X_test, y_test)*100:.1f}%")

✅ Modelo entrenado con 24129 personas
📊 Precisión en datos de prueba: 82.1%


## Paso 5: Sistema de recomendación
Esta función recibe el perfil de una persona y predice si ganará más de 50K.
Si no llega al umbral, recomienda qué cambiar.

In [5]:
def recomendar_trayectoria(perfil_usuario):
    """
    Recibe un diccionario con el perfil del usuario y devuelve una recomendación.
    Claves necesarias:
    - age (edad), education (educación), marital.status (estado civil),
    - occupation (ocupación), hours.per.week (horas/semana),
    - sex (sexo), native.country (país de origen)
    """
    # Convertir el perfil a DataFrame
    usuario_df = pd.DataFrame([perfil_usuario])
    
    # Predecir si gana más de 50K
    prediccion = model.predict(usuario_df)[0]
    
    # Obtener la probabilidad
    probabilidad = model.predict_proba(usuario_df)[0][1]
    
    if prediccion == 1:
        return f"✅ Con este perfil, tu probabilidad de ganar más de 50K es {probabilidad*100:.1f}%."
    else:
        return f"⚠️ Con este perfil, tu probabilidad de superar los 50K es solo {probabilidad*100:.1f}%. Considera mejorar tu nivel educativo o cambiar de ocupación."

## Paso 6: Probar con usuarios simulados
Creamos perfiles hipotéticos para ver qué recomienda el sistema.

In [6]:
# Usuario 1: joven con poca educación y trabajo a medio tiempo
usuario_1 = {
    "age": 25,
    "education": "HS-grad",           # Secundaria completa
    "marital.status": "Never-married", # Soltero/a
    "occupation": "Sales",             # Ventas
    "hours.per.week": 30,              # Medio tiempo
    "sex": "Male",
    "native.country": "United-States"
}

print("👤 Usuario 1 - Joven con secundaria completa:")
print(recomendar_trayectoria(usuario_1))
print()

👤 Usuario 1 - Joven con secundaria completa:
⚠️ Con este perfil, tu probabilidad de superar los 50K es solo 2.0%. Considera mejorar tu nivel educativo o cambiar de ocupación.



In [7]:
# Usuario 2: adulto con estudios universitarios y trabajo de tiempo completo
usuario_2 = {
    "age": 40,
    "education": "Bachelors",               # Universidad
    "marital.status": "Married-civ-spouse",  # Casado/a
    "occupation": "Exec-managerial",         # Gerencia
    "hours.per.week": 50,
    "sex": "Male",
    "native.country": "United-States"
}

print("👤 Usuario 2 - Adulto universitario en gerencia:")
print(recomendar_trayectoria(usuario_2))
print()

👤 Usuario 2 - Adulto universitario en gerencia:
✅ Con este perfil, tu probabilidad de ganar más de 50K es 80.0%.



In [8]:
# Usuario 3: mujer con posgrado
usuario_3 = {
    "age": 35,
    "education": "Masters",                  # Maestría
    "marital.status": "Never-married",
    "occupation": "Prof-specialty",          # Profesional especializada
    "hours.per.week": 45,
    "sex": "Female",
    "native.country": "United-States"
}

print("👤 Usuario 3 - Mujer con maestría:")
print(recomendar_trayectoria(usuario_3))

👤 Usuario 3 - Mujer con maestría:
⚠️ Con este perfil, tu probabilidad de superar los 50K es solo 16.3%. Considera mejorar tu nivel educativo o cambiar de ocupación.
